In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from preprocessing.embedding import (
    EMBEDDING_DIM,
    check_reviews,
    embeddings_to_dataframe,
    generate_embeddings,
    load_model,
    validate_embeddings,
)

In [ ]:
INPUT_FILE = PROJECT_ROOT / "data" / "processed" / "IMDB_Dataset_menor_encoded.csv"
OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "IMDB_menor_embeddings.npy"
CSV_OUTPUT_FILE = PROJECT_ROOT / "data" / "processed" / "IMDB_menor_embeddings.csv"

print("Entrada:", INPUT_FILE)
print("Saída:", OUTPUT_FILE)

In [ ]:
df = pd.read_csv(INPUT_FILE)

print("Quantidade de linhas:", len(df))
print("Quantidade de colunas:", len(df.columns))

df.head()

In [ ]:
print("Valores ausentes:", df["review"].isnull().sum())
print("Valores vazios:", (df["review"].astype(str).str.strip() == "").sum())

check_reviews(df)

In [ ]:
model = load_model()

if hasattr(model, "get_embedding_dimension"):
    dimensao = model.get_embedding_dimension()
else:
    dimensao = model.get_sentence_embedding_dimension()

print("Dimensão dos vetores:", dimensao)
print("Limite de tokens:", model.max_seq_length)

In [ ]:
exemplos = [
    "this movie was amazing, i loved it",
    "one of the best films i have ever seen",
    "terrible movie, a complete waste of time",
]

vetores = generate_embeddings(exemplos, model=model)
normas = np.linalg.norm(vetores, axis=1, keepdims=True)
similaridade = (vetores / normas) @ (vetores / normas).T

pd.DataFrame(similaridade, index=exemplos, columns=["1", "2", "3"]).round(3)

In [ ]:
embeddings = generate_embeddings(df["review"].tolist(), model=model)

In [ ]:
print("Formato da matriz:", embeddings.shape)
print("Esperado:", (len(df), EMBEDDING_DIM))
print("Valores NaN:", np.isnan(embeddings).sum())

validate_embeddings(embeddings, len(df))

In [ ]:
for i in [0, len(df) // 2, len(df) - 1]:
    individual = generate_embeddings([df.loc[i, "review"]], model=model)[0]
    print(f"Linha {i}:", np.allclose(embeddings[i], individual, atol=1e-5))

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
np.save(OUTPUT_FILE, embeddings)
print(f"Embeddings salvos em:\n{OUTPUT_FILE}")

In [ ]:
embeddings_teste = np.load(OUTPUT_FILE)

print("Formato:", embeddings_teste.shape)
print("Iguais ao gerado:", np.array_equal(embeddings, embeddings_teste))

In [ ]:
df_embeddings = embeddings_to_dataframe(embeddings, df["sentiment"])

df_embeddings.to_csv(CSV_OUTPUT_FILE, index=False)

print(f"Tabela salva em:\n{CSV_OUTPUT_FILE}")
print("Formato:", df_embeddings.shape)

df_embeddings.head()